# NB6 — Encodeur transformer figé (version PyTorch)

Cette version remplace **TensorFlow / TFAutoModel** par **PyTorch / AutoModel** pour éviter les conflits d'environnement rencontrés sous Colab.

Le principe reste identique :
- on charge `train.csv` et `test.csv`,
- on crée un **jeu de validation stratifié à partir du train**,
- on utilise un **encodeur transformer figé**,
- puis on entraîne une **petite tête MLP** au-dessus,
- et on produit les mêmes tableaux de résultats à la fin.


## Installation éventuelle

Décommente la cellule ci-dessous si nécessaire sur Colab.


In [ ]:
# %pip install -q torch transformers scikit-learn pandas numpy openpyxl

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import average_precision_score, roc_auc_score

from nlp_disaster_utils import (
    seed_everything,
    load_train_test_xy,
    stratified_validation_split,
    evaluate_probability_outputs,
    round_results,
    metric_matrix_from_results,
    save_results_bundle,
)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)


In [ ]:
DATA_DIR = "../../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False

VAL_SIZE_WITHIN_TRAIN = 0.10
RANDOM_STATE = 42

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 96
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 1

PIPELINE_NAME = "P24_FrozenDistilBERT_MLP_PyTorch"
OUTPUT_STEM = "NB6_frozen_transformer_encoder_pytorch"
RESULTS_DIR = "results"


In [ ]:
df_train, X_train_full, y_train_full, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

X_train, X_val, y_train, y_val = stratified_validation_split(
    X_train_full,
    y_train_full,
    val_size=VAL_SIZE_WITHIN_TRAIN,
    random_state=RANDOM_STATE,
)

print("Taille train      :", len(X_train))
print("Taille validation :", len(X_val))
print("Taille test       :", len(X_test))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME)
encoder.eval()

for param in encoder.parameters():
    param.requires_grad = False

encoder.to(device)
print("Encodeur chargé et gelé :", MODEL_NAME)


In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32),
        }
        return item


class FrozenTransformerMLP(nn.Module):
    def __init__(self, encoder, hidden_size, dropout1=0.3, dropout2=0.2):
        super().__init__()
        self.encoder = encoder
        self.dropout1 = nn.Dropout(dropout1)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout2)
        self.out = nn.Linear(128, 1)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        if hasattr(outputs, "last_hidden_state") and outputs.last_hidden_state is not None:
            x = outputs.last_hidden_state[:, 0, :]
        else:
            x = outputs[0][:, 0, :]

        x = self.dropout1(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout2(x)
        logits = self.out(x).squeeze(-1)
        return logits


In [ ]:
train_dataset = TweetDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset = TweetDataset(X_val, y_val, tokenizer, MAX_LEN)
test_dataset = TweetDataset(X_test, y_test, tokenizer, MAX_LEN)
train_full_dataset = TweetDataset(X_train_full, y_train_full, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
train_full_loader = DataLoader(train_full_dataset, batch_size=BATCH_SIZE, shuffle=False)

hidden_size = encoder.config.hidden_size
model = FrozenTransformerMLP(encoder=encoder, hidden_size=hidden_size).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


In [ ]:
def run_epoch(model, loader, criterion=None, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_probs = []
    all_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        if is_train:
            optimizer.zero_grad()

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels) if criterion is not None else None

        if is_train:
            loss.backward()
            optimizer.step()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.detach().cpu().numpy())

        if loss is not None:
            total_loss += float(loss.item()) * len(labels)

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    avg_loss = total_loss / len(all_labels) if criterion is not None else np.nan

    preds = (all_probs >= 0.5).astype(int)
    accuracy = (preds == all_labels).mean()

    try:
        roc_auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(all_labels, all_probs)
    except Exception:
        pr_auc = np.nan

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "probs": all_probs,
        "labels": all_labels,
    }


In [ ]:
best_state = None
best_val_loss = float("inf")
patience_counter = 0
history_rows = []

print(f"Entraînement -> {PIPELINE_NAME}")

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, criterion=criterion, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader, criterion=criterion, optimizer=None)

    history_rows.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_roc_auc": train_metrics["roc_auc"],
        "train_pr_auc": train_metrics["pr_auc"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_roc_auc": val_metrics["roc_auc"],
        "val_pr_auc": val_metrics["pr_auc"],
    })

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_metrics['loss']:.4f} | train_acc={train_metrics['accuracy']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | val_acc={val_metrics['accuracy']:.4f} | "
        f"val_roc_auc={val_metrics['roc_auc']:.4f} | val_pr_auc={val_metrics['pr_auc']:.4f}"
    )

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter > PATIENCE:
            print("Arrêt anticipé déclenché.")
            break

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history_rows)
display(round_results(history_df))


In [ ]:
final_train_metrics = run_epoch(model, train_full_loader, criterion=None, optimizer=None)
final_test_metrics = run_epoch(model, test_loader, criterion=None, optimizer=None)

result = evaluate_probability_outputs(
    name=PIPELINE_NAME,
    y_train=np.array(y_train_full),
    train_scores=final_train_metrics["probs"],
    y_test=np.array(y_test),
    test_scores=final_test_metrics["probs"],
    threshold=0.5,
)

results_df = round_results(pd.DataFrame([result]))
display(results_df)

metric_view = round_results(metric_matrix_from_results(results_df))
display(metric_view)

save_results_bundle(pd.DataFrame([result]), output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX enregistrés dans ./{RESULTS_DIR}")
